## Importações

In [1]:
!pip install -U --quiet https://github.com/iqoptionapi/iqoptionapi/archive/refs/heads/master.zip
# !pip install ta

     / 17.2 MB 15.8 MB/s 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.6/200.6 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.9/521.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.2/275.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.3/92.3 kB 7.9 MB/s eta 0:00:00


In [2]:
from iqoptionapi.stable_api import IQ_Option
import torch
import time
import pandas as pd
# import ta
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import threading

In [3]:
class DQN(nn.Module):
    """Rede Neural Profunda para a DQN (Deep Q-Network)"""

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Primeira camada totalmente conectada (input para hidden)
        self.layer1 = nn.Linear(n_observations, 128)
        # Segunda camada totalmente conectada (hidden para hidden)
        self.layer2 = nn.Linear(128, 128)
        # Terceira camada totalmente conectada (hidden para output)
        self.layer3 = nn.Linear(128, n_actions)

    # Chamado com um elemento para determinar a próxima ação, ou um lote durante a otimização
    # Retorna tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        # Aplicar a função de ativação ReLU após a primeira camada
        x = F.relu(self.layer1(x))
        # Aplicar a função de ativação ReLU após a segunda camada
        x = F.relu(self.layer2(x))
        # Passar pela terceira camada sem ativação para obter os valores Q para cada ação
        return self.layer3(x)

In [4]:
email = "sgtambrosio2019@gmail.com"
password = "YOUR_SECRET_HERE"
api = IQ_Option(email, password)
check, reason = api.connect()
print(check, reason)

True None


In [ ]:
n_observations = 10000
n_actions = 3
model = DQN(n_observations, n_actions)
model = torch.load('/content/agent_iq_option.pth',  map_location=torch.device('cpu'))
model.eval()

In [ ]:
def get_historical_data(iqoption, ativo, quantidade_total, unidade):
    """
    Obtém dados históricos de velas de um ativo em múltiplas chamadas.

    :param iqoption: Instância da API do IQ Option
    :param ativo: Ativo (ex: "EURUSD")
    :param quantidade_total: Número total de velas a serem obtidas
    :param unidade: Unidade de tempo da vela (ex: 60 para 1 minuto)
    :return: Lista completa de dados das velas
    """
    dados_completos = []
    quantidade_por_chamada = 1000  # Número de velas por chamada
    tempo_atual = int(time.time())  # Timestamp atual

    while quantidade_total > 0:
        quantidade = min(quantidade_por_chamada, quantidade_total)
        candles = iqoption.get_candles(ativo, unidade, quantidade, tempo_atual)
        if not candles:
            break
        dados_completos.extend(candles)
        tempo_atual = candles[-1]['from'] - 1  # Atualiza o timestamp para pegar velas anteriores
        quantidade_total -= quantidade

    return dados_completos

In [ ]:
active = 'EURUSD'
unidade = 60     # segundos
quantidade = 10000
dados = get_historical_data(api, active, quantidade, unidade)
dados = pd.DataFrame(dados)

In [ ]:
dados['from'] = pd.to_datetime(dados['from'], unit='s')
dados["Date"] = pd.to_datetime(dados["from"]).dt.date
dados.drop(["from", "id", "at", "to"], axis=1, inplace=True)

In [ ]:
dados.head()

In [ ]:
# Exemplo: Calcular a Média Móvel Simples (SMA)
def calculate_sma(data, window=100):
    sma = ta.trend.sma_indicator(data['close'], window)
    return sma

In [ ]:
# Exemplo: Calcular o RSI
def calculate_rsi(data, window=100):
    rsi = ta.momentum.rsi(data['close'], window)
    return rsi

In [ ]:
def check_order_result(order_id):
    # Verificar o resultado da ordem
    result = api.check_win_digital_v2(order_id)
    if result:
        print(f"Ordem {order_id} Ganho: {result}")
    else:
        print(f"Ordem {order_id} não foi bem-sucedida. Perda: {result}")

In [ ]:
def select_action(output, active='EURUSD', duration=1, und=1000):
    banca = api.get_balance()
    predict = torch.argmax(output).item()
    amount = banca / und

    if predict == 1:
        action = 'call'
        status, order_id = api.buy_digital_spot_v2(active, amount, action, duration)
        if status:
            print(f"Compra realizada com sucesso! ID da ordem: {order_id}")
            time.sleep(duration * 60)
            retunrs = check_order_result(order_id)
    elif predict == 2:
        action = 'put'
        status, order_id = api.buy_digital_spot_v2(active, amount, action, duration)
        if status:
            print(f"Venda realizada com sucesso! ID da ordem: {order_id}")
            retunrs = check_order_result(order_id)

In [ ]:
def update_and_predict():
    active = 'EURUSD'
    unidade = 60     # segundos
    quantidade = 10000  # Número de velas a serem obtidas

    while True:
        # Obtendo os dados históricos
        dados = get_historical_data(api, active, quantidade, unidade)

        # Converter para DataFrame
        data = pd.DataFrame(dados)

        # Converter timestamps para datetime
        data['from'] = pd.to_datetime(data['from'], unit='s')

        # Remover colunas desnecessárias
        data.drop(["from", "id", "at", "to", "volume"], axis=1, inplace=True)

        # Calcular SMA e RSI
        data['SMA'] = calculate_sma(data)
        data['RSI'] = calculate_rsi(data)

        # Remover valores NaN
        data.dropna(inplace=True)

        # Converter para array numpy
        data_array = np.array(data.astype(float), dtype="float32")

        # Realizando uma previsão com o modelo
        example_input = torch.tensor(data_array[-1])  # Pegue a última linha dos dados
        output = model(example_input)

        # Imprimir previsão do modelo
        print("Previsão do modelo:", torch.argmax(output).item())

        # Selecionar ação baseada na previsão
        select_action(output)

        print(f"Aguardando {30} segundos...")
        time.sleep(30)

In [ ]:
update_and_predict()